# Dashboard

In [1]:
import pandas as np
import pandas as pd

In [74]:
Db=pd.read_excel(r"C:\Users\Asus\Downloads\Dashboard_Gogo.xlsx")
print(Db)   

   agent_name              customer_name             car_detail  \
0      สมหญิง        นาย สมศักดิ์ ใจดี 1     Honda City กข 1001   
1       สมชาย       นาง สมศรี มีทรัพย์ 1    Toyota Vios คง 2001   
2      วิภาดา            นาย มานะ อดทน 1    Isuzu D-Max จฉ 3001   
3        มานพ          นาง วีณา สวยงาม 1  Nissan Almera ชซ 4001   
4        สุดา           นาย ปิติ ยินดี 1        Mazda 2 ญฎ 5001   
..        ...                        ...                    ...   
95     สมหญิง         นาง นารี ร่ำรวย 10    Honda Civic ดต 6010   
96      สมชาย     นาย ชาติชาย กล้าหาญ 10   Toyota Hilux ถท 7010   
97     วิภาดา      นางสาว อรทัย ใจงาม 10          MG ZS ธน 8010   
98       มานพ  นาย สมเกียรติ ยิ่งใหญ่ 10    Ford Ranger บป 9010   
99       สุดา       นาง จินตนา น่ารัก 10   Suzuki Swift ผฝ 0010   

    overdue_installment  total_debt  total_interest  total_fine  \
0                     1       10500             200          50   
1                     2       21000             500         1

# Model classifire

In [85]:
Cf=pd.read_excel(r"C:\Users\Asus\Downloads\json_call_logs_1000_labeled.xlsx")
Cf

,Label,json_data
0,Customer in noisy environment,"{""outbound_id"": ""8308444535"", ""phone_number"": ""062455998..."
1,Customer not convenient to talk,"{""outbound_id"": ""8550666260"", ""phone_number"": ""064956220..."
2,Customer refused to pay,"{""outbound_id"": ""8332490247"", ""phone_number"": ""061289977..."
3,Customer interested in debt restructuring,"{""outbound_id"": ""8302774430"", ""phone_number"": ""063128244..."
4,Customer requested human agent,"{""outbound_id"": ""8339072443"", ""phone_number"": ""067689507..."
...,...,...
995,No answer – call back later,"{""outbound_id"": ""8760959353"", ""phone_number"": ""066911161..."
996,Customer refused to talk to bot,"{""outbound_id"": ""8183639001"", ""phone_number"": ""068449048..."
997,Customer has hardship situation,"{""outbound_id"": ""8179121000"", ""phone_number"": ""060613736..."
998,Language barrier,"{""outbound_id"": ""8606275746"", ""phone_number"": ""063611151..."


In [17]:
Cf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   json_data  100 non-null    object
dtypes: object(1)
memory usage: 928.0+ bytes


In [91]:
counts = Cf_result["classified_code"].value_counts()
total  = len(Cf_result)

print(f"{'Status Code':<30} {'Count':>6}  {'%':>6}")
print("-" * 45)
for code, count in counts.items():
    print(f"{code:<30} {count:>6}  {count/total*100:>5.1f}%")
print("-" * 45)
print(f"{'TOTAL':<30} {total:>6}  100.0%")

Status Code                     Count       %
---------------------------------------------
Refused_to_Pay                    154   15.4%
Bad_Connection                     77    7.7%
Not_Convenient_to_Talk             77    7.7%
Hardship_Case                      77    7.7%
Requested_Human_Agent              77    7.7%
Promise_to_Pay                     77    7.7%
Promise_to_Pay_No_Date             77    7.7%
No_Answer                          77    7.7%
Refused_to_Talk_to_Bot             77    7.7%
Language_Barrier                   77    7.7%
Unknown                            77    7.7%
Phone_Off                          76    7.6%
---------------------------------------------
TOTAL                            1000  100.0%


In [90]:
Cf_result.to_excel("classified_resultsv16.xlsx", index=False)

In [89]:
"""
classify_v4.py  — แก้ปัญหา 4 จุดจากผลลัพธ์จริง
─────────────────────────────────────────────────────────────────────────────
FIX 1  รูป3/4  Refused_to_Pay → ควรเป็น Requested_Human_Agent / Hardship_Case
       สาเหตุ: first-match ติด Refused จากบรรทัดแรก
       แก้:   ใช้ last-match wins ทุก layer

FIX 2  รูป5   Customer_Silent → ควรเป็น Promise_to_Pay_No_Date
       สาเหตุ: "สะดวก" อยู่ใน IGNORE แต่บรรทัดถัดไปลูกค้าพูด "จ่าย แต่ไม่รู้วันไหน"
               ซึ่งถูก join เป็น customer_text แต่โค้ดเดิม first-match ไม่ถึง Promise
       แก้:   last-match + เพิ่ม pattern "จ่าย.*แต่.*ไม่.*รู้.*วัน|จ่าย.*ไม่.*รู้.*วัน"

FIX 3  รูป6   ไม่มีผู้รับสาย/โทรออกครั้งที่ 1 → ควรเป็น No_Answer
       สาเหตุ: pattern ไม่ครอบคลุม "โทรออกครั้งที่ 1 ไม่มีผู้รับสาย"
       แก้:   เพิ่ม pattern + ตรวจ full log เป็น layer พิเศษ

FIX 4  Customer_Silent ที่ไม่มี customer lines เลย
       แก้:   ถ้า status=completed และ customer_lines ว่าง → Customer_Silent
              แต่ถ้า full log มี pattern อื่น → ใช้ pattern นั้นก่อน
─────────────────────────────────────────────────────────────────────────────
"""

import re
import json
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# PATTERNS พิเศษ
# ─────────────────────────────────────────────────────────────────────────────

IGNORE_PATTERN = re.compile(
    r"^(สะดวก|ครับ|ค่ะ|คะ|อ่ะ|อะ|ได้|โอเค|ok|okay|hello|สวัสดี|ฮัลโหล"
    r"|yes|no|ใช่|เออ|อือ|หือ|ไม่ทราบ|ไม่รู้)$",
    re.IGNORECASE,
)

# ตรวจเฉพาะ customer lines / action เท่านั้น (ไม่ full log)
SILENT_PATTERN = re.compile(
    r"เงียบ|ไม่พูด|ไม่ตอบ|ไม่มีเสียง|ลูกค้าเงียบ"
    r"|silent|customer.*silent|no.*interaction|dead.*air|inactive",
    re.IGNORECASE,
)

# ─────────────────────────────────────────────────────────────────────────────
# RULES  (pattern, label_en, code)
# ─────────────────────────────────────────────────────────────────────────────
RULES = [
    # ── Phone Off ────────────────────────────────────────────────────────────
    (
        r"ปิดเครื่อง|โทรไม่ติด|นอกเขต|ติดต่อไม่ได้|ไม่มีสัญญาณ"
        r"|สายถูกตัด|เบอร์นี้ปิดใช้|ระงับสัญญาณ|ถูกระงับบริการ"
        r"|เบอร์ถูกยกเลิก|หมายเลขนี้เลิกใช้แล้ว|ไม่สามารถติดต่อปลายทาง"
        r"|เครือข่ายล่ม|ซิมถูกระงับ|สัญญาณขัดข้อง|เครื่องดับ"
        r"|voicemail|voice\s*mail|phone.*off|switched.*off"
        r"|number.*disconnected|service.*suspended|carrier.*error"
        r"|not.*reachable|unavailable|subscriber.*unavailable",
        "Phone is turned off", "Phone_Off",
    ),

    # ── No Answer  [FIX 3: เพิ่ม pattern โทรออกครั้งที่ / ไม่มีผู้รับสาย] ──
    (
        r"ไม่รับสาย|วางสาย|ไม่ได้รับสาย|ไม่มีคนรับ|สายไม่ว่าง"
        r"|ปล่อยให้ดัง|ไม่มีการกดรับ|ตัดสายทิ้ง|สายหลุดทันที"
        r"|ปล่อยสาย|ไม่กดรับ|ไม่รับโทรศัพท์|ไม่มีการตอบรับ"
        r"|ไม่มีผู้รับสาย|โทรออกครั้งที่\s*\d|ระบบจะโทรใหม่|โทรแล้วไม่มีคนรับ"
        r"|missed\s*call|no\s*pickup|didn.t.*answer|ignored.*call"
        r"|no\s*answer|hung\s*up|no\s*response|ring.*no.*answer"
        r"|auto.*disconnect|call.*dropped|no.*pickup.*response",
        "No answer – call back later", "No_Answer",
    ),

    # ── Refused to Talk to Bot ───────────────────────────────────────────────
    (
        r"ไม่คุยกับบอท|ไม่อยากคุยกับบอท|ไม่คุยกับระบบ|ไม่คุยกับเครื่อง"
        r"|ไม่เอาเครื่องโทร|ไม่ต้องการระบบนี้|ไม่คุยกับโปรแกรม"
        r"|วางเพราะเป็นระบบ|สายอัตโนมัติไม่เอา"
        r"|ไม่คุยกับai|ไม่เอาระบบอัตโนมัติ|ไม่ต้องการbot"
        r"|automated.*call.*reject|system.*call.*no"
        r"|don.t.*want.*ai|hate.*bot|robot.*call.*no"
        r"|refuse.*bot|won.t.*talk.*bot|not.*talk.*bot|ระบบอัตโนมัติ.*ไม่",
        "Customer refused to talk to bot", "Refused_to_Talk_to_Bot",
    ),

    # ── Requested Human Agent ────────────────────────────────────────────────
    (
        r"ขอเจ้าหน้าที่|คุยกับคนจริง|ขอคุยกับเจ้าหน้าที่|ขอพูดกับคน|ขอโอนสาย"
        r"|ให้พนักงานติดต่อมา|ขอคุยฝ่ายบริการ|อยากคุยฝ่ายดูแลลูกค้า"
        r"|ติดต่อเจ้าหน้าที่กลับ|ขอเจ้าหน้าที่โทรหา"
        r"|อยากคุยกับคน|ให้คนโทรมา|ขอคุยกับพนักงาน"
        r"|ให้เจ้าหน้าที่ติดต่อกลับ|อยากคุยกับแอดมิน|โอนให้คน"
        r"|human\s*agent|speak.*person|real\s*person|transfer.*agent"
        r"|support.*agent|customer.*service.*call|call.*me.*back.*human"
        r"|ดิฉัน.*จะประสาน|จะประสานงาน.*เจ้าหน้าที่|ติดต่อกลับ.*เจ้าหน้าที่",
        "Customer requested human agent", "Requested_Human_Agent",
    ),

    # ── Debt Restructuring  [FIX 1/4: เพิ่ม "สนใจ" ตอบรับ Bot] ─────────────
    (
        r"ปรับโครงสร้าง|ปรับหนี้|ขอผ่อน|ผ่อนชำระ|ขอต่อรอง|ขอลดดอก"
        r"|หมุนเงินไม่ทัน|สภาพคล่องติดขัด|รายรับไม่พอรายจ่าย"
        r"|ภาระเยอะ|ค่าใช้จ่ายสูง|ขาดสภาพคล่อง"
        r"|ขอปรับยอด|ขอผ่อนยาว|ขอเลื่อนงวด|ขอลดค่างวด"
        r"|ขอแบ่งจ่าย|แบ่งจ่าย|โครงการ.*ปรับ|สนใจ.*โครงการ|สนใจ.*ปรับ"
        r"|debt\s*restruc|restructur|negotiate|installment\s*plan"
        r"|liquidity.*issue|over.*expense|financial.*burden"
        r"|extend.*term|lower.*installment|reduce.*payment",
        "Customer interested in debt restructuring", "Hardship_Case",
    ),

    # ── Hardship (เหตุการณ์ยากลำบาก) ────────────────────────────────────────
    (
        r"น้ำท่วม|ตกงาน|เจ็บป่วย|ป่วย|อุบัติเหตุ|พิการ|เสียชีวิต"
        r"|ไฟไหม้|ภัยพิบัติ|วิกฤต|รายได้ลด|ธุรกิจแย่|ขายของไม่ได้|ขาดทุน"
        r"|บ้านพัง|ของหาย|lost.*job|seriously\s*ill"
        r"|hardship|disaster|accident|unemployed"
        r"|income.*drop|financial.*problem|cashflow.*issue",
        "Customer has hardship situation", "Hardship_Case",
    ),

    # ── Refused to Pay  (ต้องอยู่ก่อน Promise) ──────────────────────────────
    (
        r"ไม่ยอมจ่าย|ไม่มีเงิน|ไม่มีเงินจริง|ไม่สามารถจ่าย|จ่ายไม่ได้"
        r"|ไม่จ่าย|ปฏิเสธ|ยังไม่ไหว|ไม่คิดจะชำระ|ยังไม่จ่ายแน่นอน"
        r"|ไม่พร้อมชำระ|เลี่ยงจ่าย|ไม่รับผิดชอบยอด"
        r"|ไม่มีจ่าย|ยังไม่พร้อมจ่าย|ไม่คิดจะจ่าย"
        r"|ไม่มีงบ|ติดขัด.*เงิน|เงินไม่พอ"
        r"|refuse.*pay|won.t.*pay|cannot.*pay|can.t.*pay|no.*money"
        r"|avoid.*payment|decline.*payment|refuse.*settlement"
        r"|skip.*payment|refuse.*payment",
        "Customer refused to pay", "Refused_to_Pay",
    ),

    # ── Promise to Pay with Date  (ต้องอยู่ก่อน No Date) ────────────────────
    (
        r"ยอมจ่าย.*วัน|จะจ่าย.*วัน|จ่าย.*วันที่|นัดชำระ|โอน.*วันที่"
        r"|อีก\s*\d+\s*วันจะจ่าย|ภายใน\s*\d+\s*วัน"
        r"|ก่อนสิ้นเดือน|หลังเงินเดือนออก"
        r"|จ่าย.*(จันทร์|อังคาร|พุธ|พฤหัส|ศุกร์|เสาร์|อาทิตย์)"
        r"|จ่าย.*พรุ่งนี้|จ่าย.*มะรืน|จ่าย.*อาทิตย์หน้า"
        r"|สิ้นเดือน|ต้นเดือน|กลางเดือน|ภายในสัปดาห์"
        r"|after.*salary|in\s*\d+\s*days|before.*month.*end"
        r"|end.*of.*month|next.*week|within.*week"
        r"|promise.*date|will.*pay.*(monday|tuesday|wednesday"
        r"|thursday|friday|tomorrow|\d{1,2}[\/\-]\d{1,2})",
        "Customer promised to pay with date", "Promise_to_Pay",
    ),

    # ── Promise to Pay No Date  [FIX 2: เพิ่ม "จ่าย.*แต่.*ไม่.*รู้.*วัน"] ──
    (
        r"ยอมจ่าย|จะจ่าย|จะโอน|จะชำระ|เดี๋ยวจ่าย|จ่ายเอง"
        r"|กำลังจะจ่าย|มีแผนจะจ่าย|รอจ่าย"
        r"|เดี๋ยวจัดการให้|กำลังดำเนินการจ่าย|รอเคลียร์ยอด"
        r"|จะทยอยจ่าย|มีแผนเคลียร์หนี้"
        r"|จ่าย.*แต่.*ไม่.*รู้.*วัน|จ่าย.*ไม่.*รู้.*วันไหน|จ่าย.*ไม่.*แน่ใจ.*วัน"
        r"|processing.*payment|arranging.*payment|settle.*soon"
        r"|plan.*to.*pay|intend.*to.*pay|promise.*pay|will.*pay|i.ll.*pay",
        "Customer promised to pay (no date)", "Promise_to_Pay_No_Date",
    ),

    # ── Noisy Environment ────────────────────────────────────────────────────
    (
        r"เสียงดัง|เสียงดังมาก|ตอนนี้เสียงดัง|อยู่ที่เสียงดัง"
        r"|เสียงรบกวน|คุยไม่รู้เรื่อง|เสียงขาดๆหายๆ"
        r"|เสียงขาด|สายกระตุก|ฟังไม่ต่อเนื่อง|เสียงสะดุด|คุยติดๆดับๆ"
        r"|สัญญาณไม่ดี|ได้ยินไม่ชัด|สัญญาณแย่|เสียงไม่ชัด"
        r"|noisy|loud|bad.*signal|poor.*connection|can.t.*hear"
        r"|audio.*cut|connection.*unstable|static.*noise|voice.*breaking"
        r"|signal.*weak|background.*noise",
        "Customer in noisy environment", "Bad_Connection",
    ),

    # ── Not Convenient to Talk ───────────────────────────────────────────────
    (
        r"ไม่สะดวก|อยู่ข้างทาง|ติดธุระ|ติดประชุม|กำลังขับรถ"
        r"|ติดภารกิจ|กำลังเดินทาง|อยู่ระหว่างทำธุระ"
        r"|ขอเลื่อนคุย|ยังไม่สะดวกรับสาย|ติดสาย|กำลังทำงาน"
        r"|ไม่พร้อมคุย|ขอเวลา|อยู่ในที่ประชุม|ไม่ว่าง|ยุ่งอยู่"
        r"|โทรมาใหม่|โทรกลับ"
        r"|not.*convenient|busy|in.*meeting|driving|call.*later"
        r"|on.*the.*way|currently.*busy|occupied|engaged|bad.*time",
        "Customer not convenient to talk", "Not_Convenient_to_Talk",
    ),

    # ── Language Barrier ─────────────────────────────────────────────────────
    (
        r"ภาษาถิ่น|ภาษาอีสาน|ภาษาเหนือ|ภาษาใต้|พูดภาษาอื่น"
        r"|สื่อสารไม่เข้าใจ|ฟังไม่ออกเลย|พูดคนละภาษา"
        r"|ต้องใช้ล่าม|ไม่ถนัดภาษาไทย|พูดไทยไม่ได้|ฟังไทยไม่ออก"
        r"|ไม่เข้าใจภาษา|ภาษาต่างประเทศ|ภาษาจีน|ภาษาพม่า|ภาษาลาว"
        r"|language\s*barrier|doesn.t.*speak.*thai|speaks.*only"
        r"|need.*translator|not.*fluent|cannot.*communicate"
        r"|cannot.*understand|communication.*problem",
        "Language barrier", "Language_Barrier",
    ),
]

# Compile ล่วงหน้า
COMPILED_RULES = [
    (re.compile(pat, re.IGNORECASE | re.DOTALL), lbl, code)
    for pat, lbl, code in RULES
]

# ─────────────────────────────────────────────────────────────────────────────
# Helper: last-match wins  (FIX 1 — แก้ปัญหา first-match ติด Refused)
# ─────────────────────────────────────────────────────────────────────────────

def _last_match(text: str) -> tuple | None:
    """วิ่งทุก rule แล้วคืน match ที่อยู่ท้ายสุดของ text"""
    best_end, best_label, best_code = -1, None, None
    for compiled, label, code in COMPILED_RULES:
        for m in compiled.finditer(text):
            if m.end() > best_end:
                best_end, best_label, best_code = m.end(), label, code
    return (best_label, best_code) if best_label else None


def _first_match(text: str) -> tuple | None:
    """first-match สำหรับ full_log layer (ความเร็ว)"""
    for compiled, label, code in COMPILED_RULES:
        if compiled.search(text):
            return label, code
    return None


# ─────────────────────────────────────────────────────────────────────────────
# Helper: แยก customer lines
# ─────────────────────────────────────────────────────────────────────────────

def _extract_customer_lines(conv: str) -> list[str]:
    lines = []
    for line in conv.splitlines():
        stripped = line.strip()
        if stripped.lower().startswith("customer"):
            text = stripped.split(":", 1)[-1].strip()
            if text and not IGNORE_PATTERN.match(text):
                lines.append(text)
    return lines


# ─────────────────────────────────────────────────────────────────────────────
# classify
# ─────────────────────────────────────────────────────────────────────────────

def classify(record: dict) -> dict:
    action = str(record.get("action", "") or "")
    conv   = str(record.get("conversation_log", "") or "")
    status = str(record.get("status", "") or "")

    conv = conv.replace("\\n", "\n")

    customer_lines    = _extract_customer_lines(conv)
    last_line         = customer_lines[-1] if customer_lines else ""
    all_customer_text = " ".join(customer_lines)

    # ════════════════════════════════════════════════════════════════════════
    # LAYER 1  บรรทัดสุดท้ายที่ลูกค้าพูด  →  last-match wins
    #          FIX 1/2: จับ "สนใจ" / "จ่ายแต่ไม่รู้วัน" ที่ท้ายบทสนทนา
    # ════════════════════════════════════════════════════════════════════════
    if last_line:
        result = _last_match(last_line)
        if result:
            return {"label": result[0], "code": result[1], "layer": 1}

    # ════════════════════════════════════════════════════════════════════════
    # LAYER 2  action field  →  last-match + Silent แยก
    # ════════════════════════════════════════════════════════════════════════
    if action:
        result = _last_match(action)
        if result:
            return {"label": result[0], "code": result[1], "layer": 2}
        if SILENT_PATTERN.search(action):
            return {"label": "Customer silent", "code": "Customer_Silent", "layer": 2}

    # ════════════════════════════════════════════════════════════════════════
    # LAYER 3  ทุกบรรทัดลูกค้ารวมกัน  →  last-match wins
    #          FIX 2: "จ่าย แต่ไม่รู้วันไหน" อยู่บรรทัดอื่นจะจับได้ที่นี่
    # ════════════════════════════════════════════════════════════════════════
    if all_customer_text:
        result = _last_match(all_customer_text)
        if result:
            return {"label": result[0], "code": result[1], "layer": 3}
        if SILENT_PATTERN.search(all_customer_text):
            return {"label": "Customer silent", "code": "Customer_Silent", "layer": 3}

    # ════════════════════════════════════════════════════════════════════════
    # LAYER 4  full log  →  first-match (ไม่รวม Silent)
    #          FIX 3: "โทรออกครั้งที่ 1 ไม่มีผู้รับสาย" จะจับได้ที่นี่
    # ════════════════════════════════════════════════════════════════════════
    if conv:
        result = _first_match(conv)
        if result:
            return {"label": result[0], "code": result[1], "layer": 4}

    # ════════════════════════════════════════════════════════════════════════
    # LAYER 5  status field fallback
    #          FIX 4: completed + ไม่มี customer lines → Customer_Silent
    # ════════════════════════════════════════════════════════════════════════
    if status == "no_answer":
        return {"label": "No answer – call back later", "code": "No_Answer", "layer": 5}
    if status == "completed" and not customer_lines:
        return {"label": "Customer silent", "code": "Customer_Silent", "layer": 5}

    return {"label": "Unknown", "code": "Unknown", "layer": 0}


# ─────────────────────────────────────────────────────────────────────────────
# classify_dataframe
# ─────────────────────────────────────────────────────────────────────────────

def classify_dataframe(df: pd.DataFrame, col: str = "json_data") -> pd.DataFrame:
    labels, codes, layers = [], [], []
    total = len(df)

    for i, raw in enumerate(df[col], 1):
        try:
            record = json.loads(raw) if isinstance(raw, str) else (raw or {})
        except (json.JSONDecodeError, TypeError):
            record = {"conversation_log": str(raw)}

        r = classify(record)
        labels.append(r["label"])
        codes.append(r["code"])
        layers.append(r["layer"])

        if i % 50 == 0 or i == total:
            print(f"  [{i}/{total}] done")

    out = df.copy()
    out["classified_label"] = labels
    out["classified_code"]  = codes
    out["match_layer"]      = layers
    return out


# ─────────────────────────────────────────────────────────────────────────────
# print_summary
# ─────────────────────────────────────────────────────────────────────────────

def print_summary(df: pd.DataFrame, code_col: str = "classified_code") -> None:
    counts  = df[code_col].value_counts()
    total   = counts.sum()
    unknown = counts.get("Unknown", 0)
    known   = total - unknown
    sep     = "─" * 52
    sep2    = "═" * 52

    print(f"\n{sep2}")
    print(f"  สรุปผล Classification  (ทั้งหมด {total:,} rows)")
    print(f"{sep2}")
    print(f"  {'Status Code':<28} {'Count':>5}  {'%':>5}")
    print(sep)
    for code, cnt in counts.items():
        pct  = cnt / total * 100
        bar  = "█" * max(1, int(pct / 2))
        warn = "  ⚠" if code == "Unknown" else ""
        print(f"  {code:<28} {cnt:>5}  {pct:>4.1f}%  {bar}{warn}")
    print(sep)
    print(f"  {'รู้จัก':<28} {known:>5}  {known/total*100:>4.1f}%")
    if unknown:
        print(f"  {'Unknown ⚠':<28} {unknown:>5}  {unknown/total*100:>4.1f}%")
    print(f"{sep2}\n")

    if "match_layer" in df.columns:
        lmap = {
            1: "last_line",
            2: "action",
            3: "all_customer",
            4: "full_log",
            5: "status_fallback",
            0: "Unknown",
        }
        print("  Layer breakdown:")
        for lyr, cnt in df["match_layer"].value_counts().sort_index().items():
            print(f"    Layer {lyr} ({lmap.get(lyr,'?'):<18}): {cnt:>5} rows  ({cnt/total*100:.1f}%)")
        print()


# ─────────────────────────────────────────────────────────────────────────────
# Run  — วาง cell นี้ต่อจาก cell ที่โหลด Cf แล้ว run ได้เลย
# ─────────────────────────────────────────────────────────────────────────────

print("Classifying...")
Cf_result = classify_dataframe(Cf, col="json_data")
print_summary(Cf_result)

Cf_result.to_json("classified_results.json", orient="records",
                  force_ascii=False, indent=2)
print("Saved → classified_results.json")

Classifying...
  [50/1000] done
  [100/1000] done
  [150/1000] done
  [200/1000] done
  [250/1000] done
  [300/1000] done
  [350/1000] done
  [400/1000] done
  [450/1000] done
  [500/1000] done
  [550/1000] done
  [600/1000] done
  [650/1000] done
  [700/1000] done
  [750/1000] done
  [800/1000] done
  [850/1000] done
  [900/1000] done
  [950/1000] done
  [1000/1000] done

════════════════════════════════════════════════════
  สรุปผล Classification  (ทั้งหมด 1,000 rows)
════════════════════════════════════════════════════
  Status Code                  Count      %
────────────────────────────────────────────────────
  Refused_to_Pay                 154  15.4%  ███████
  Bad_Connection                  77   7.7%  ███
  Not_Convenient_to_Talk          77   7.7%  ███
  Hardship_Case                   77   7.7%  ███
  Requested_Human_Agent           77   7.7%  ███
  Promise_to_Pay                  77   7.7%  ███
  Promise_to_Pay_No_Date          77   7.7%  ███
  No_Answer                 

In [84]:
Cf_result

,json_data,classified_label,classified_code,match_layer
0,"{""conversation_log"":""Bot: สวัสดี\nCustomer: รายได้ลดลง\n...",Customer has hardship situation,Hardship_Case,3
1,"{""conversation_log"":""Bot: สวัสดี\nCustomer: เงินไม่พอ\nB...",Customer interested in debt restructuring,Hardship_Case,1
2,"{""conversation_log"":""Bot: สวัสดี\nCustomer: สะดวก\nBot: ...",Customer promised to pay (no date),Promise_to_Pay_No_Date,1
3,"{""conversation_log"":""Bot: โทรออกครั้งที่ 1 ไม่มีผู้รับสา...",No answer – call back later,No_Answer,4
